# 4. Test of the selected model and comparison with the MPE manuscript

1. The model with the lowest validation error is selected (or the one named in `MODEL`).
2. It reconstructs Â from (D1, D2) for the 1,000 test families and for the 380 families with k = 2 of the MPE manuscript. The true ancestor is not seen.
3. **Comparison with the original ancestor A:** exact recovery, e(A, Â), ε(A, Â) and n(Â) − n(A), by event density, as in the MPE manuscript.
4. **Comparison with the MPE result:** on the test set against the alignment-based pipeline run on the same families (notebook 2), and on the 380 MPE families against the results recorded for the manuscript, family by family.
5. **Distance estimates:** d_est(Â, Dᵢ) from the model's ancestor under the match-reward and the edit-weight scorings, against d_hist(A, Dᵢ); sign counts and summary statistics of δ_rel.

Steps 1 to 4 need the GPU for a few minutes. Step 5 is CPU work (about 1 to 2 hours in one session) and is parallel and resumable in the same way as notebook 2.

In [ ]:
#@title Setup: fetch the code and choose where results are kept
REPO_URL = "https://github.com/AtheerMatroud/TandemRepeats"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}

import os, sys, subprocess, zipfile, io
if not os.path.exists("/content/TandemRepeats/AI/tr_ai"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/TandemRepeats"])
if not os.path.exists("/content/TandemRepeats/AI/tr_ai"):
    from google.colab import files
    print("The AI folder was not found in the repository. Upload AI.zip instead.")
    up = files.upload(); name = next(iter(up))
    os.makedirs("/content/TandemRepeats", exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall("/content/TandemRepeats")
AI = "/content/TandemRepeats/AI"
sys.path.insert(0, AI)
os.environ["PYTHONPATH"] = AI + ":" + os.environ.get("PYTHONPATH", "")
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/tandem_repeats/ai_experiment"
else:
    ROOT = "/content/ai_experiment"
os.makedirs(ROOT, exist_ok=True)
import json, glob, time, numpy as np, pandas as pd
print("code:", AI, "| results:", ROOT, "| CPU cores:", os.cpu_count())


In [ ]:
#@title Parallel settings
# One session already uses every CPU core. To go faster, open this notebook in several
# Colab sessions at once and give each a different PART (0, 1, ..., N_PARTS - 1).
# All sessions write to the same Drive folder; finished shards are skipped, so an
# interrupted session can simply be run again.
PART = 0     #@param {type:"integer"}
N_PARTS = 1  #@param {type:"integer"}


In [ ]:
#@title Select the model and reconstruct the ancestors of the test families
MODEL = "auto"  #@param ["auto", "aligntag", "bilstm", "transformer"]
import torch
from tr_ai import simdata, evaluate as ev
from tr_ai.infer import load_model, predict
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cands = {p.split("/")[-2]: json.load(open(p))["best"] for p in glob.glob(f"{ROOT}/models/full/*/done.json")}
display(pd.DataFrame(cands).T.sort_values("eps"))
name = min(cands, key=lambda k: cands[k]["eps"]) if MODEL == "auto" else MODEL
print("selected model:", name)
fams = {f["id"]: f for f in simdata.load(f"{ROOT}/families/shard_*.jsonl.gz")}
splits = json.load(open(f"{ROOT}/splits.json"))
SETS = {"test": [fams[i] for i in splits["test"]], "mpe": simdata.load(f"{ROOT}/mpe_grid/s*.jsonl.gz")}
PRED = {}
for key, fs in SETS.items():
    path = f"{ROOT}/pred_{name}_{key}.json"
    if not os.path.exists(path):
        model, _ = load_model(name, f"{ROOT}/models/full/{name}/ckpt_best.pt", device)
        t0 = time.time(); preds = predict(model, fs, device, progress=True)
        json.dump({"ids": [f["id"] for f in fs], "Ahat": preds, "seconds": time.time() - t0}, open(path, "w"))
    PRED[key] = json.load(open(path))["Ahat"]
    print(key, len(PRED[key]), "ancestors reconstructed")
ACC = {key: pd.DataFrame([ev.accuracy_row(f, p) for f, p in zip(SETS[key], PRED[key])]) for key in SETS}


In [ ]:
#@title Accuracy against the true ancestor, and comparison with the alignment-based pipeline
base_test = pd.DataFrame(ev.read_shards(f"{ROOT}/baseline_test"))
base_mpe = ev.mpe_frame(f"{AI}/data/mpe_k2_results.csv")
BASE = {"test": base_test, "mpe": base_mpe}
TITLE = {"test": "test set (1,000 families)", "mpe": "the 380 families with k = 2 of the MPE manuscript"}
tables = {}
for key in SETS:
    common = set(ACC[key].id) & set(BASE[key].id)
    if len(common) < len(ACC[key]): print(f"note: baseline available for {len(common)} of {len(ACC[key])} families so far; tables use those")
    A, B = ACC[key][ACC[key].id.isin(common)], BASE[key][BASE[key].id.isin(common)]
    a = ev.accuracy_table(A).assign(method=f"learned model ({name})")
    b = ev.accuracy_table(B).assign(method="alignment-based (MPE)")
    tables[key] = pd.concat([a, b]).set_index(["method", "event density"])
    print("Accuracy of the reconstructed ancestor:", TITLE[key]); display(tables[key])
    pr = pd.DataFrame({d: ev.paired(A[m(A)], B) for d, m in (("low", lambda x: x.low == 1), ("high", lambda x: x.low == 0),
                                                              ("all", lambda x: x.low >= 0))}).T
    print("Paired comparison of eps on the same families (a = learned model, b = alignment-based)"); display(pr.round(4))
    tables[key].to_csv(f"{ROOT}/table_accuracy_{key}.csv")


In [ ]:
#@title Figures: error by t and by s, excess length, heatmaps
import matplotlib.pyplot as plt
for key in SETS:
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    for df, lab in ((ACC[key], f"learned model ({name})"), (BASE[key], "alignment-based (MPE)")):
        g = df.groupby("t"); ax[0].plot(g.eps.mean().index, 100 * g.eps.mean().values, marker="o", label=lab)
        g = df.groupby("s"); ax[1].plot(g.eps.mean().index, 100 * g.eps.mean().values, marker="o", label=lab)
        g = df.groupby("t"); ax[2].plot(g.excess.mean().index, g.excess.mean().values, marker="o", label=lab)
    ax[0].set_title("mean relative error (%) against t"); ax[0].set_xlabel("t (5t events per descendant)")
    ax[1].set_title("mean relative error (%) against s"); ax[1].set_xlabel("s (about 50s copies in A)")
    ax[2].set_title("mean n(Ahat) - n(A) against t"); ax[2].set_xlabel("t")
    for a in ax: a.grid(alpha=.3); a.legend()
    fig.suptitle(TITLE[key]); plt.tight_layout(); plt.savefig(f"{ROOT}/fig_accuracy_{key}.png", dpi=150); plt.show()
    fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    vmax = 100 * max(ACC[key].groupby(["s", "t"]).eps.mean().max(), BASE[key].groupby(["s", "t"]).eps.mean().max())
    for a, (df, lab) in zip(ax, ((ACC[key], f"learned model ({name})"), (BASE[key], "alignment-based (MPE)"))):
        h = 100 * df.pivot_table(index="t", columns="s", values="eps", aggfunc="mean")
        im = a.imshow(h.values, origin="lower", aspect="auto", vmin=0, vmax=vmax, extent=[h.columns.min() - .5, h.columns.max() + .5, h.index.min() - .5, h.index.max() + .5])
        ss = np.linspace(2, 20, 100); a.plot(ss, np.sqrt(2 * 50 * ss * np.log(2)) / 5, color="orange")
        a.set_ylim(0.5, 20.5); a.set_title(lab); a.set_xlabel("s")
    ax[0].set_ylabel("t"); fig.colorbar(im, ax=ax, label="mean relative error (%)")
    fig.suptitle(TITLE[key] + ": low event density lies below the orange curve")
    plt.savefig(f"{ROOT}/fig_heatmap_{key}.png", dpi=150, bbox_inches="tight"); plt.show()


In [ ]:
#@title Distance estimates from the model's ancestor (CPU, parallel, resumable)
for key in SETS:
    items = sorted(zip(SETS[key], PRED[key]), key=lambda x: -len(x[0]["A"]))
    ev.run_sharded(ev.model_distance_family, items, f"{ROOT}/dist_{name}_{key}", shard_size=10, part=PART, n_parts=N_PARTS)


In [ ]:
#@title delta_rel: sign counts and summary statistics (format of the MPE manuscript)
out = []
for key in SETS:
    mod = pd.DataFrame(ev.read_shards(f"{ROOT}/dist_{name}_{key}"))
    print(f"\n=== {TITLE[key]}: {len(mod)} of {len(SETS[key])} families done ===")
    B = BASE[key][BASE[key].id.isin(set(mod.id))]; mod = mod[mod.id.isin(set(B.id))]
    src = {f"learned model ({name})": ev.edges_frame(mod, "hat"), "alignment-based (MPE)": ev.edges_frame(B, "hat"),
           "true ancestor A": ev.edges_frame(B, "true")}
    for sc in ev.SCORINGS:
        for who, e in src.items():
            sign, stats = ev.drel_tables(e, sc)
            both = sign.merge(stats, on="event density").assign(set=key, scoring=ev.LABEL[sc], ancestor=who)
            out.append(both)
        print(f"delta_rel under the {ev.LABEL[sc]} scoring")
        display(pd.concat(out[-len(src):]).set_index(["ancestor", "event density"]).drop(columns=["set", "scoring"]))
    mod.drop(columns=["Ahat"]).to_csv(f"{ROOT}/model_{name}_{key}.csv", index=False)
pd.concat(out).to_csv(f"{ROOT}/table_delta_rel.csv", index=False)
ACC["test"].to_csv(f"{ROOT}/accuracy_{name}_test.csv", index=False); ACC["mpe"].to_csv(f"{ROOT}/accuracy_{name}_mpe.csv", index=False)
print("All result files are in", ROOT)
